# 06 — Full RAG Pipeline
Build a complete Retrieval-Augmented Generation system: ingest documents → index → retrieve → answer.

In [ ]:
# !pip install langchain langchain-openai langchain-community faiss-cpu tiktoken pypdf python-dotenv
import os
from dotenv import load_dotenv
load_dotenv()


## 1. Prepare Documents

In [ ]:
from langchain_core.documents import Document

# Simulated knowledge base (replace with real PDFs using PyPDFLoader)
docs = [
    Document(page_content="LangChain was created by Harrison Chase and first released in October 2022 as an open-source Python framework.", metadata={"source": "langchain_history.txt"}),
    Document(page_content="LCEL stands for LangChain Expression Language. It was introduced in 2023 and uses the pipe operator | to compose Runnables.", metadata={"source": "lcel_docs.txt"}),
    Document(page_content="LangGraph is a library built on top of LangChain for building stateful, multi-actor LLM applications with cyclical graphs.", metadata={"source": "langgraph_docs.txt"}),
    Document(page_content="LangSmith is LangChain's observability and evaluation platform. It lets you trace, debug, and test LLM chains in production.", metadata={"source": "langsmith_docs.txt"}),
    Document(page_content="Vector stores like FAISS and Chroma store text embeddings and support approximate nearest-neighbour search for semantic retrieval.", metadata={"source": "vectorstores.txt"}),
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines a retriever with an LLM so the model can answer questions grounded in external documents.", metadata={"source": "rag_overview.txt"}),
    Document(page_content="Prompt templates in LangChain allow you to define reusable prompts with named placeholders that are filled at runtime.", metadata={"source": "prompts.txt"}),
    Document(page_content="Agents in LangChain use the ReAct pattern: the LLM alternates between Thought, Action, and Observation until it produces a Final Answer.", metadata={"source": "agents_docs.txt"}),
]
print(f"Loaded {len(docs)} documents")


## 2. Split Documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split at 300 chars with 50-char overlap to preserve cross-sentence context
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks   = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")
for i, c in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} [{c.metadata['source']}]:")
    print(c.page_content)


## 3. Create Embeddings and Vector Store

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Embed all chunks and store in FAISS (in-memory)
embedder     = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore  = FAISS.from_documents(chunks, embedding=embedder)

# Save to disk so you don't re-embed every time
vectorstore.save_local("../data/faiss_index")
print("Vector store created and saved.")


## 4. Load Vector Store and Create Retriever

In [ ]:
# Load from disk (skip embedding step on subsequent runs)
vectorstore = FAISS.load_local(
    "../data/faiss_index",
    embeddings=embedder,
    allow_dangerous_deserialization=True
)

# Retriever: returns top-3 most relevant chunks
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Test retrieval
test_results = retriever.invoke("What is LCEL?")
print(f"Retrieved {len(test_results)} chunks:")
for r in test_results:
    print(f"  - [{r.metadata['source']}] {r.page_content[:80]}")


## 5. Build the RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

def format_docs(docs):
    """Concatenate retrieved chunks into a single context string."""
    return "\n\n".join(
        f"[Source: {d.metadata['source']}]\n{d.page_content}" for d in docs
    )

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the question using ONLY the context below.\n"
     "If the answer is not in the context, say 'I don\'t have that information.'\n\n"
     "Context:\n{context}"),
    ("human", "{question}")
])

rag_chain = (
    {
        "context":  retriever | format_docs,  # retrieves + formats docs
        "question": RunnablePassthrough()     # passes question through unchanged
    }
    | rag_prompt
    | llm
    | parser
)

print("RAG chain ready.")


## 6. Query the RAG System

In [ ]:
questions = [
    "Who created LangChain and when?",
    "What is the difference between LangGraph and LangSmith?",
    "How does the ReAct pattern work in agents?",
    "What is RAG and why is it useful?",
    "What is the capital of France?",  # should trigger 'I don't have that information'
]

for q in questions:
    print(f"Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}\n")


## 7. RAG with Conversation Memory

In [ ]:
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import MessagesPlaceholder

# Memory-aware RAG prompt
memory_rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the context below.\n\nContext:\n{context}"),
    MessagesPlaceholder(variable_name="history"),  # injects past turns
    ("human", "{question}")
])

memory = ConversationBufferWindowMemory(k=4, return_messages=True, memory_key="history")

def rag_with_memory(question: str) -> str:
    history = memory.load_memory_variables({})["history"]
    context = format_docs(retriever.invoke(question))
    response = (memory_rag_prompt | llm | parser).invoke({
        "context":  context,
        "question": question,
        "history":  history
    })
    memory.save_context({"input": question}, {"output": response})
    return response

# Multi-turn conversation
print(rag_with_memory("What is LangChain?"))
print(rag_with_memory("Who built it?"))
print(rag_with_memory("What other tools does the same ecosystem provide?"))


---
**Key takeaway:** RAG solves the context window limit — instead of stuffing all documents into the prompt, you retrieve only the relevant chunks at query time.